# Chapter 5 — Decoders in Action

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzafarooq/advanced-rag-from-scratch/blob/main/colab_original_notebooks/Chapter_5.ipynb)

Companion code for **Chapter 5** of *Build an Advanced RAG Application (From Scratch)*.

We turn from retrieval (Chapter 4) to generation. The decoder is the **producer** of the answer — and how we *prompt* it determines the quality of what comes out.

This notebook walks through four prompting styles:

1. **Basic** — single-line ask
2. **Structured** — explicit sections and constraints
3. **Few-shot** — show the model the pattern by example
4. **Chain-of-Thought (CoT)** — make the model reason step by step

By the end, we apply CoT to **analyze hotel search results** — which is exactly the seam Chapter 6 plugs retrieval into.



## 1. Setup

This notebook calls the OpenAI API. Add your `OPENAI_API_KEY` in colab secrets before running.


In [ ]:
!pip install openai --quiet

In [ ]:
# Configure the OpenAI client and define a simple text-generation helper.

import os
from openai import OpenAI

# Set up the OpenAI API key
from google.colab import userdata
from IPython.display import display, Markdown # Import the necessary modules

client = OpenAI(
    # This is the default and can be omitted
    api_key= userdata.get('OPENAI_API_KEY'),
)


# Step 1: Define the prompt for the decoder model
prompt = "Generate a short story about Paris"
# The prompt is a text string that provides context and instructions to the model

# Step 2: Set the parameters for the ChatGPT-4 API
model_engine = "gpt-4o"
# Specify the model engine to use (e.g., "gpt-4" for ChatGPT-4)
temperature = 0.2
# Control the randomness of the generated text (higher values = more random)
max_tokens = 1000

# Step 3: Generate the decoder model output

def generate_text(prompt, model="gpt-4o"):
  chat_completion = client.chat.completions.create(
      messages=[
          {
              "role": "user",
              "content": prompt,
          }
      ],
      model=model,
      temperature=temperature,
      max_tokens=max_tokens,

  )


  # Call the OpenAI API to generate the decoder model output based on the provided parameters

  # Extract the generated text from the API response
  return chat_completion.choices[0].message.content
# Access the generated text from the API response
# The generated text is stored in the 'text' field of the json output

# 2. Basic vs. structured prompting

The same model, the same topic, two prompts. The difference in answer quality is entirely on us.


In [ ]:
generate_text(prompt)

In [ ]:
# Define a basic prompt and a more structured version for comparison.

# Basic prompt (less effective)
basic_prompt = "Tell me about Paris hotels"

# More structured prompt (better)
structured_prompt = """
Provide information about hotels in Paris, including:
- Location considerations
- Price ranges
- Popular areas for tourists
- Transportation access
Please format the response in clear sections.
Give complete answer in the tokens, do not cut off halfway
"""

In [ ]:
display(Markdown(generate_text(basic_prompt))) # Use display and Markdown to render

In [ ]:
display(Markdown(generate_text(structured_prompt))) # Use display and Markdown to render

### Try it on a different topic

The pattern transfers — be specific about *what you want, in what shape*.


In [ ]:
Basic_Prompt = "Tell me about climate change"

Engineered_Prompt = """Provide a comprehensive analysis of climate change, focusing on:
1. Current scientific consensus
2. Major environmental impacts
3. Mitigation strategies
Format this as a structured report with clear headings and evidence-based conclusions."""


In [ ]:
display(Markdown(generate_text(Basic_Prompt))) # Use display and Markdown to render

In [ ]:
display(Markdown(generate_text(Engineered_Prompt))) # Use display and Markdown to render

## 3. Few-shot prompting

When you can't fully specify the format in words, *show* it. Two or three labeled examples are usually enough.


In [ ]:
# Redefine the helper to include few-shot examples inside the prompt.

from openai import OpenAI
client = OpenAI(
    # This is the default and can be omitted
    api_key= userdata.get('OPENAI_API_KEY'),
)


def generate_text(prompt, model="gpt-4o"):


    few_shot_examples = """
    Example 1:
    User: Tell me about New York hotels
    AI: The Plaza Hotel in New York is an iconic luxury hotel located at Fifth Avenue and Central Park South. It offers timeless elegance, world-class dining, and top-tier hospitality.

    Example 2:
    User: Tell me about Tokyo hotels
    AI: The Park Hyatt Tokyo is a prestigious hotel known for its stunning skyline views, sophisticated atmosphere, and exceptional dining options. Located in Shinjuku, it provides a tranquil retreat in the heart of the city.

    Now, following the same pattern, provide a response:
    User: {prompt}
    AI:
    """.strip()

    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "user", "content": few_shot_examples.format(prompt=prompt)}
        ],
        model=model,
        temperature=0.7,
        max_tokens=200,
    )

    return chat_completion.choices[0].message.content

# Example usage:
prompt = "Tell me about Paris hotels"
response = generate_text(prompt)
print(response)

## 4. Chain-of-Thought prompting

CoT asks the model to *show its work* — reasoning step by step before giving the final answer. It tends to:

- Improve correctness on multi-step problems
- Make wrong answers easier to spot (the bad reasoning is visible)
- Cost more tokens (it produces more text)


In [ ]:
# Redefine the helper again to use chain-of-thought style prompting.

def generate_text(prompt, model="gpt-4o"):


    cot_prompt = """
    Let's think step by step.

    Example 1:
    User: Tell me about New York hotels
    AI: First, I will select a famous hotel in New York. The Plaza Hotel is a well-known luxury hotel. Then, I will highlight its key attributes, such as its historic significance, prime location near Central Park, and world-class dining. Finally, I will summarize why it stands out.
    Answer: The Plaza Hotel in New York is an iconic luxury hotel located at Fifth Avenue and Central Park South. It offers timeless elegance, world-class dining, and top-tier hospitality.

    Example 2:
    User: Tell me about Tokyo hotels
    AI: First, I will choose a renowned hotel in Tokyo. One example is The Park Hyatt Tokyo. Then, I will describe its notable aspects, including its stunning skyline views, sophisticated atmosphere, and location in Shinjuku. Lastly, I will explain why it is a preferred choice for visitors.
    Answer: The Park Hyatt Tokyo is a prestigious hotel known for its stunning skyline views, sophisticated atmosphere, and exceptional dining options. Located in Shinjuku, it provides a tranquil retreat in the heart of the city.

    Now, following the same structured reasoning approach, provide a response:
    User: {prompt}
    AI:
    """.strip()

    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "user", "content": cot_prompt.format(prompt=prompt)}
        ],
        model=model,
        temperature=0.7,
        max_tokens=500,
    )

    return chat_completion.choices[0].message.content

# Example usage:
prompt = "Tell me about Paris hotels"
response = generate_text(prompt)
print(response)

## 5. CoT applied to retrieval results — the bridge into Chapter 6

This is the prompt pattern Chapter 6 uses to summarize FAISS / Qdrant search results into a grounded answer.

For demonstration we paste in some example retrieval output. In Chapter 6 the `results` string comes from the search pipeline programmatically.


In [ ]:
query = 'Hotel near the Louvre with great food nearby.'

result = """
Top hotel with similar reviews using FAISS:
1. Grand Hotel du Palais Royal
Review: Great hotel located in the best part of town just next to the Louvre. Walking distance to the Louvre, Norte Dame, and shops. Very clean and the service was wonderful. Even though we didn’t speak French everyone at the hotel was very nice and helpful. Almost all of the staff spoke English. The spa is small though which can make things awkward if using the spa amenities.
Distance: 0.7877

2. Hotel Malte - Astotel
Review: Excellent hotel all round.  Ideal location for The Louvre, city centre shops, and plenty of metro stations within a few minutes walk to get to other areas of Paris.  There is no restaurant for evening meals and lunch and no bar in the hotel but that doesn't matter as there are numerous restaurants and cafes close by so you don't have to go far to eat out.  Nothing was too much trouble for the staff and they were all so polite and helpful.  The room was very comfortable.  The bed was as good as you could wish for and the en-suite bathroom had plenty of room.  The room and the hotel in general was spotlessly clean.  A nice touch was the mini bar fridge with soft drinks and water in the room at no extra charge, as was the food buffet with hot drinks and soft drinks that was
Distance: 0.7877

3. Hotel Moliere
Review: Travelled with another couple and we stayed for 3 nights and an additional night after visiting the countryside.  Highly recommend.  Excellent location near Louvre and other key sites.  We really enjoyed the helpfulness of the staff and the hospitality there including the delicious breakfast and refreshments offered. Rooms have good amenities but are small and varied.  We didn't love the bathroom layout with a glass wall that didn't offer much privacy.
Distance: 0.7869

4. La Maison Favart
Review: Great location - peaceful and very central - opposite the Opera House.  Fifteen minutes walk down to the Louvre and many good restaurants close by.  Very comfortable beds and pretty breakfast room.  Front of House staff couldn't be more helpful especially with directions and great restaurants !
Distance: 0.7856

5. Hotel Square Louvois
Review: What a great hotel. The staff was over the top nice. Every single one of them. Location was great. Room was clean and nice.   Went with 3 teen boys and the adjoining room was perfect in every way.   Close to metro walking distance to Louvre and dozens of restaurants.
Distance: 0.7829

6. Hotel Malte - Astotel
Review: Brilliant hotel, clean, comfortable, great location just a short walk to the Louvre. The open bar in the afternoon is fantastic, we could refresh with coffee and pastries throughout the day at any of their locations, which is such a brilliant idea. Staff were friendly and welcoming. Cannot fault the hotel in any way.
Distance: 0.7771

7. Hotel La Comtesse
Review: Great hotel and caring staff, and they are always ready to provide anything needed. Great location for visiting the Eiffel Tower, Louvre museum, royal palaces, and great restaurants.  Rooms are Very well appointed, with very comfortable beds/bedding. Good selection of breakfast items, from hot to cold. Great place to stay!
Distance: 0.7756

8. Hotel Moliere
Review: One of the quietest rooms I ever slept in in Paris. Staff is very attentive. The room is very clean, no bedbugs! Breakfast is prepared with lots of love for details. If you want to visit the Louvre, well, this is a very convenient hotel to stay at. Enjoy.
Distance: 0.7719

9. Hotel Square Louvois
Review: Stayed 3 nights in their superior room. Great location - just 10mins or so by foot to the Louvre and Tuileries Garden Christmas Market. There are also lots of restaurants nearby - many French bistros as well as Asian restaurants (Telegraph wrote that Rue St. Anne is one of Paris' best streets for Asian cuisine). Many metro lines are accessible from the hotel (opera, quatre septembre, pyramides, and Auber to catch the train to Disneyland).  The bathroom is not the most spacious (and the shower could be improved), but the room has everything you need, from bathrobes, slippers, high quality toilet paper to a nespresso machine.   Didn't get a chance to enjoy their complimentary tea time from 4 to 6pm, but there were free madeleines (which we needed for an early train ride to
Distance: 0.7708

10. Novotel Paris Les Halles
Review: This was a great hotel in a wonderful location. We stayed for 4 nights with our two young girls. The staff was very friendly and almost everyone spoke English. One of the best things was the amazing breakfast. We had planned on trying to get out to cafes in the morning but with the kids it was so much easier to just eat at the hotel since our rate had meals included. Which ended up being fine since the food was so great. Nice location with very short walk to Louvre, metro stop, Rue Montorgueil, children's garden. Our girls loved the free ice cream in the afternoon! The only down side to the room is that the bathroom walls and doors are opaque. So if someone goes to the bathroom in the middle of the night it lights up the whole room.
Distance: 0.7671

11. Hotel Malte - Astotel
Review: Charming and wonderful hotel in rue de Richelieu. Excellent staff - friendly and helpful. Rooms are clean. Quick check-in and out. Clean rooms. Close to the Louvre museum and other areas of interest. Paris is a walking city but lots of taxis and Uber available. Very good breakfast and free snacks from 2:00 pm onwards, all kinds of nice restaurants nearby.
Distance: 0.7557

12. Hotel Malte - Astotel
Review: Beautiful hotel! Close to the Tuileries Christmas Market -  15 min walk and same distance or less to the Louvre and many shopping places.  The staff was amazing, water and other refreshments were always waiting in my room. Breakfast was a plus - worth the additional cost. Hope to return again! 🇫🇷❤️
Distance: 0.7534

13. Hotel Square Louvois
Review: This is a lovely hotel, beautifully decorated. Not far from Palais Royal, the Tuilleries & Louvre. Centrally located but also quiet. Cannot say enough positive things about the staff who were helpful and welcoming. They happily chatted with me, despite my broken French, so that I could get some practice. Breakfast was delicious.
Distance: 0.7533

14. Hotel du Danube Saint Germain
Review: Beautiful and whimsical, the staff are incredibly friendly and the location is great. You turn the corner and are about 5 minutes from the Louvre, and there are plenty of yummy restaurants nearby. The rooms are full of character and are very Parisian. My husband and I stayed for our first wedding anniversary and it definitely felt special. Thank you for a lovely stay!
Distance: 0.7506

15. Hotel Malte - Astotel
Review: Perfect hotel with amazing staffs, all good rooms, great breakfast and perfect location. Walkable to all spot like louvre, 30min to Concorde and so on with lot of multi cusine restaurants around. Loved it.
Distance: 0.7461

16. Hotel Square Louvois
Review: Outstanding boutique hotel, friendly and welcoming staff, beautifully decorated rooms and public spaces, located on a fairly quiet street but with plenty of bistros nearby and within walking distance of the Louvre. Thank you to the staff for a wonderful experience. We’ll stay here again next time we’re in Paris and would definitely recommend.
Distance: 0.7453

17. Hotel Moliere
Review: We had a wonderful recent stay at this hotel and can highly recommend it.  The location couldn't be beat -- minutes from the Louvre and the Tulleries as well as the metro and restaurants -and the room was tasteful, spacious, spotless, and comfortable.  Staff was very helpful both before and during our stay.  A winner!
Distance: 0.7427

18. Best Western Plus La Demeure
Review: An excellent hotel with top class staff.  Breakfast had hot and cold food and drinks of a high standard and the staff were welcoming and friendly.  This hotel is in a great location to visit Paris with many locations being within walking distance.  Highly recommended.
Distance: 0.7424

19. Hotel Tourisme Avenue
Review: Hotel was in a great location, many places to eat in close proximity to hotel and walking distances to the Eiffel Tower. Room set up was great for family of 3. The hotel was clean and staff was friendly.
Distance: 0.7387

20. Grand Hotel du Palais Royal
Review: The hotel is in a great location, near the Louvre and walking distance to restaurants.  The staff at this hotel go above and beyond, very professional and with great suggestions/recommendations. Rooms are very clean and the pillows/bedding super comfortable. Will definitely return!
Distance: 0.7376

21. Hotel Malte - Astotel
Review: A nice boutique hotel on a quiet street, at easy walking distance to the Louvre, Tuileries, Palais Royal and Opera Garnier. Small but comfortable rooms that are well outfitted with modern bathrooms.  Front desk staff are very helpful. Breakfast has a variety of options for people with different tastes. Free snacks and drinks in the lobby from 2 pm till 2 am are a nice bonus. The area has several restaurants and cafes.  If possible, I would choose a room on the second floor (American third floor) or above. The rooms on the first floor (American second floor) are not the quietest for light sleepers or those who like to sleep in, as you hear the sounds from the lobby and the dining area downstairs.
Distance: 0.7373

22. Hotel Tourisme Avenue
Review: This hotel was wonderful. The location is great near most attractions In Paris. It’s next to several wonderful cafe’s and several grocery stores.  the staff was very nice the rooms were clean and beds were very comfortable. The bathrooms were beautiful.
Distance: 0.7372

23. Hotel Joke - Astotel
Review: Very comfortable hotel with great staff. Varied and healthy breakfast buffet. Smart to visit the  city center. Good space in the room. Located near Moulin Rouge and Pigalle district. Very good both for couples and families  Beverages and snacks offered
Distance: 0.7356

24. Hotel Malte - Astotel
Review: This hotel is the best. All the staff of lobby desk and restaurant do their best. Thank you for them. Especially Anastasiia,  thank you. The Louvre Museum and Musee de l'Orangerie are really close. You can walk to the Eiffel or take the Metro all at once. I love the dust-free shiny bathroom, the breakfast with fresh orange juice and bread (you can steam your eggs), the free snack bar in the lobby all afternoon when you come for a short break, and the staff's kind and precise handling of the work.   (The room was the lowest grade, so it was small, but if you want a room with a large room, book a room with a higher grade.)
Distance: 0.7355

25. Hotel Europe Saint Severin
Review: Love the location, room comfort and cleanliness, staff friendliness, and atmosphere. Quaint rooms - especially the ones with little balconies. Right near the St. Michel metro, near Notre Dame, St. Germain de Pres, Ile St. Loius, Luxembourg Gardens, the Marais and the Louvre.
Distance: 0.7348
"""


In [ ]:
# Build an analysis prompt that asks the model to reason over retrieved reviews.

prompt = f"""

  Analyze the following hotel search results and answer these questions using a chain of thought:

  1. Which hotel best matches the user's query: '{query}'?  Consider location, food options, and overall experience.
  2. What are the pros and cons of the top 3 hotels?  Support your answer with quotes from the reviews.
  3.  Are there any recurring themes or patterns in the positive/negative reviews?
  4. Summarize the top 3 hotels in bullet points highlighting their strengths and weaknesses.
  5. Based on the provided data, provide an overall recommendation to the user.

  Hotel Search Results:
  {result}

  """

In [ ]:
prompt

In [ ]:
# Wrap the retrieval-analysis pattern in a reusable helper function.

# prompt: a chain of thought prompt which can be used for results from the query


# Assuming 'hotel_query', and 'result' are defined as in the previous example

def analyze_hotel_recommendations(query, results,model="gpt-4o"):
  """Analyzes hotel recommendations based on a query and provided results.

  Args:
    query: The user's hotel search query.
    results: A string containing hotel recommendations and reviews.

  Returns:
    A Markdown string summarizing the analysis.
  """

  # Chain of thought prompting
  prompt = f"""

  Analyze the following hotel search results and answer these questions using a chain of thought:

  1. Which hotel best matches the user's query: '{query}'?  Consider location, food options, and overall experience.
  2. What are the pros and cons of the top 3 hotels?  Support your answer with quotes from the reviews.
  3.  Are there any recurring themes or patterns in the positive/negative reviews?
  4. Summarize the top 3 hotels in bullet points highlighting their strengths and weaknesses.
  5. Based on the provided data, provide an overall recommendation to the user.

  Hotel Search Results:
  {results}
  """

  # Use OpenAI API to get the analysis

  chat_completion = client.chat.completions.create(
        messages=[
            {"role": "user", "content": prompt}
        ],
        model=model,
        temperature=0.7,
        max_tokens=1500,
    )

  analysis = chat_completion.choices[0].message.content

  return analysis


# Example usage:
analysis = analyze_hotel_recommendations(query, result)
display(Markdown(analysis))


## What's next

In Chapter 6 we replace the hard-coded `results` block with a live FAISS / Qdrant search and stream the LLM's answer back to the user — a complete RAG loop.
